# Notebook 1: The b-value Stability Atlas

This notebook maps both the Gutenberg-Richter b-value and its temporal volatility across the globe using 25 years of USGS earthquake data. The b-value describes the relative proportion of small to large earthquakes in a region; deviations from the canonical b ≈ 1.0 and temporal instability in b can reveal tectonic stress changes, fluid injection effects, or volcanic unrest. We compute b-values on a 2°×2° global grid, measure their coefficient of variation (CV_b) over sliding time windows, examine the relationship between b-value and depth, and track regional b-value evolution for six tectonically distinct zones.

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, ".")
from src.catalog import estimate_mc
from src.gutenberg_richter import (
    estimate_b_value, compute_bvalue_grid, compute_cv_b, rolling_b_value
)
from src.spatial import assign_cells
from src.plotting import (
    setup_style, save_figure, plot_global_map, plot_bvalue_volatility_map
)

setup_style()

In [2]:
df = pd.read_csv("data/earthquake_catalog_global.csv")
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
print(f"Loaded {len(df):,} events")
print(f"Date range: {df['time'].min()} \u2192 {df['time'].max()}")
print(f"Magnitude range: {df['mag'].min():.1f} \u2192 {df['mag'].max():.1f}")

Loaded 681,450 events
Date range: 2000-01-01 01:19:26.990000+00:00 → 2025-12-31 23:52:11.327000+00:00
Magnitude range: 0.7 → 9.1


## 1.1 Global b-value Map

In [3]:
# Compute b-values on 2°×2° grid
bvalue_grid = compute_bvalue_grid(df, cell_size=2.0, min_events=50)
print(f"Cells with valid b-values: {len(bvalue_grid)}")
print(f"b-value range: {bvalue_grid['b_value'].min():.2f} \u2192 {bvalue_grid['b_value'].max():.2f}")
print(f"Median b: {bvalue_grid['b_value'].median():.2f}")

Cells with valid b-values: 709
b-value range: 0.40 → 2.18
Median b: 1.09


In [4]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_global_map(
    bvalue_grid["cell_lat"], bvalue_grid["cell_lon"], bvalue_grid["b_value"],
    cmap="RdYlBu_r", vmin=0.6, vmax=1.4,
    label="b-value (MLE)", title="Global Gutenberg-Richter b-value (2000\u20132025)",
    ax=ax
)
save_figure(fig, "01_global_bvalue_map")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_14630/3661931277.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.2 Temporal Volatility (CV_b)

In [5]:
cv_grid = compute_cv_b(df, cell_size=2.0, window_years=3, stride_years=1,
                        min_events_per_window=50, min_events_total=200)
print(f"Cells with CV_b: {len(cv_grid)}")
print(f"CV_b range: {cv_grid['cv_b'].min():.3f} \u2192 {cv_grid['cv_b'].max():.3f}")
print(f"Median CV_b: {cv_grid['cv_b'].median():.3f}")

Cells with CV_b: 303
CV_b range: 0.000 → 0.594
Median CV_b: 0.108


In [6]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_bvalue_volatility_map(cv_grid["cell_lat"], cv_grid["cell_lon"], cv_grid["cv_b"], ax=ax)
save_figure(fig, "01_global_cv_b_map")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_14630/55170868.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.3 b-value vs. Depth

In [7]:
# Merge depth info with b-value grid
df_cells = assign_cells(df, cell_size=2.0)
depth_by_cell = df_cells.groupby(["cell_lat", "cell_lon"])["depth"].median().reset_index()
depth_by_cell.columns = ["cell_lat", "cell_lon", "median_depth"]

bv_depth = bvalue_grid.merge(depth_by_cell, on=["cell_lat", "cell_lon"])

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(bv_depth["median_depth"], bv_depth["b_value"],
                c=bv_depth["n_events"], cmap="viridis", s=15, alpha=0.6,
                norm=plt.matplotlib.colors.LogNorm())
plt.colorbar(sc, ax=ax, label="Number of events")
ax.set_xlabel("Median Depth (km)")
ax.set_ylabel("b-value")
ax.set_title("b-value vs. Depth (2\u00b0\u00d72\u00b0 cells)")
ax.set_xlim(0, 700)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
save_figure(fig, "01_bvalue_vs_depth")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_14630/3495311201.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.4 Regional b-value Time Series

In [8]:
regions = {
    "Japan Trench": {"lat": (30, 45), "lon": (135, 150)},
    "San Andreas": {"lat": (32, 37), "lon": (-122, -115)},
    "Oklahoma": {"lat": (33.5, 37.5), "lon": (-100, -94.5)},
    "Sumatra": {"lat": (-5, 10), "lon": (92, 106)},
    "Iceland": {"lat": (63, 67), "lon": (-25, -13)},
    "Yellowstone": {"lat": (43, 46), "lon": (-112, -109)},
}

fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
axes_flat = axes.flatten()

for i, (name, bounds) in enumerate(regions.items()):
    ax = axes_flat[i]
    mask = (
        (df["latitude"] >= bounds["lat"][0]) & (df["latitude"] <= bounds["lat"][1]) &
        (df["longitude"] >= bounds["lon"][0]) & (df["longitude"] <= bounds["lon"][1])
    )
    region_df = df[mask].copy()

    if len(region_df) < 200:
        ax.set_title(f"{name} (insufficient data)")
        continue

    mc = estimate_mc(region_df["mag"].values)
    rolling = rolling_b_value(
        region_df["time"].values, region_df["mag"].values,
        mc=mc, window_size=200, step=50
    )

    if len(rolling) > 0:
        ax.plot(
            pd.to_datetime(rolling["center_time"]), rolling["b_value"],
            color="#2A9D8F", linewidth=1
        )
        ax.fill_between(
            pd.to_datetime(rolling["center_time"]),
            rolling["b_value"] - rolling["b_std"],
            rolling["b_value"] + rolling["b_std"],
            alpha=0.2, color="#2A9D8F"
        )
        ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)

    ax.set_title(f"{name} (Mc={mc:.1f}, n={len(region_df):,})")
    ax.set_ylabel("b-value")

axes_flat[-2].set_xlabel("Date")
axes_flat[-1].set_xlabel("Date")
fig.suptitle(
    "Regional b-value Time Series (rolling 200-event windows)",
    fontsize=14, fontweight="bold"
)
fig.tight_layout()
save_figure(fig, "01_regional_bvalue_timeseries")
plt.show()

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_14630/157847126.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This notebook constructed a global atlas of Gutenberg-Richter b-values and their temporal stability. Key observations:

- The global median b-value clusters near the canonical value of 1.0, but substantial regional variation exists across tectonic settings.
- Subduction zones and volcanic regions tend to show elevated b-values, while stable continental interiors trend lower.
- The coefficient of variation (CV_b) highlights regions where b-values are temporally unstable, which may indicate transient stress perturbations, fluid injection, or evolving fault networks.
- The b-value vs. depth analysis reveals systematic trends consistent with increasing differential stress at depth.
- Regional time series show that some zones (e.g., Oklahoma) exhibit pronounced b-value shifts coinciding with known changes in industrial activity, while purely tectonic regions maintain more stable b-values over time.

These spatial and temporal b-value patterns form the baseline for identifying anomalous seismicity signatures in subsequent notebooks.